<a href="https://colab.research.google.com/github/benAloo/olilo/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fg-data-profiling pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.3/400.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import data_profiling
from data_profiling import ProfileReport
from google.colab import files

In [9]:
uploaded = files.upload()

Saving car_details_v3.csv to car_details_v3.csv


In [10]:
df = pd.read_csv('car_details_v3.csv')

In [11]:
profile = ProfileReport(df, title="Profiling Report", explorative=True)
profile.to_file('report.html')
files.download('report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 28.67it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
df.shape

(8128, 13)

In [14]:
df.dtypes

,0
name,object
year,int64
selling_price,int64
km_driven,int64
fuel,object
seller_type,object
transmission,object
owner,object
mileage,object
engine,object


In [15]:
df.isna().sum()

,0
name,0
year,0
selling_price,0
km_driven,0
fuel,0
seller_type,0
transmission,0
owner,0
mileage,221
engine,221


In [23]:
df.head(3)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0


In [17]:
df.columns = (
    df.columns
    .str.replace(' ', '_')
    .str.lower()
    .str.strip()
)

In [18]:
missing = df.isna().mean() * 100
print(missing[missing > 0].sort_values(ascending=False))

torque       2.731299
mileage      2.718996
engine       2.718996
seats        2.718996
max_power    2.645177
dtype: float64


- If your data has 40-50% missing values, check the data pipeline

In [20]:
n_dupes = df.duplicated().sum()
n_dupes

np.int64(1202)

In [22]:
# How the duplicates look like
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(3)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
1977,Audi Q3 2.0 TDI Quattro Premium Plus,2017,2825000,22000,Diesel,Dealer,Automatic,First Owner,15.73 kmpl,1968 CC,174.33 bhp,380Nm@ 1750-2500rpm,5.0
7324,Audi Q3 2.0 TDI Quattro Premium Plus,2017,2825000,22000,Diesel,Dealer,Automatic,First Owner,15.73 kmpl,1968 CC,174.33 bhp,380Nm@ 1750-2500rpm,5.0
2129,Audi Q5 3.0 TDI Quattro,2014,1850000,76131,Diesel,Individual,Automatic,First Owner,13.22 kmpl,2967 CC,241.4 bhp,580Nm@ 1400-3250rpm,5.0


In [24]:
df = df.drop_duplicates()

In [25]:
df.shape

(6926, 13)

#### Handling missing values

- Replace categorical values with "unknown"
- Numeric columns replace with median, but not mean because if there are outliers it will impact mean values

In [31]:
df = df.copy()

# Columns that require string cleaning before numeric conversion and median imputation
string_numeric_cols = ['max_power', 'mileage', 'engine', 'torque']

# Apply specific cleaning for each column if needed, then convert to numeric and fill NaNs
for col in string_numeric_cols:
    cleaned_series = df[col].astype(str) # Start with string conversion for all

    if col == 'max_power':
        # Remove ' bhp' from max_power
        cleaned_series = cleaned_series.str.replace(' bhp', '', regex=False)
    elif col == 'mileage':
        # Remove ' kmpl' and ' km/kg' from mileage
        cleaned_series = cleaned_series.str.replace(' kmpl', '', regex=False).str.replace(' km/kg', '', regex=False)
    elif col == 'engine':
        # Remove ' CC' from engine
        cleaned_series = cleaned_series.str.replace(' CC', '', regex=False)
    elif col == 'torque':
        # Extract the numeric part before units or special characters (e.g., '190Nm@ 2000rpm' -> '190')
        # Corrected regex with single backslashes
        cleaned_series = cleaned_series.str.extract(r'(\d+\.?\d*)', expand=False)

    df[col] = pd.to_numeric(cleaned_series, errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Columns that are already numeric but might have NaNs to be filled with median
simple_numeric_cols = ['seats']
for col in simple_numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns
categorical_cols = ['name', 'fuel', 'seller_type', 'transmission', 'owner']

df[categorical_cols] = df[categorical_cols].fillna('Unknown')

In [32]:
print(df.isna().sum()[df.isna().sum() > 0])

torque    6926
dtype: int64
